# Single-Compartment Neurons

`braincell` builds every neuron on the Hodgkin–Huxley (HH) framework. Two concrete cell classes share this framework through inheritance:

- `HHTypedNeuron`: the abstract base class that provides the interface for HH-type neurons.
- `SingleCompartment`: a single-compartment (point) neuron model, suitable for neurons with simple structure.
- `MultiCompartment`: a spatially detailed, multi-compartment neuron model — covered in the [Cell](cell.ipynb) tutorial.

Together these components form a modeling system that spans from local membrane regions to the complete neuronal structure. Through inheritance they share a unified modeling interface and, with the acceleration and computational power of `brainstate`, provide strong support for neuronal modeling at the cellular level.

This tutorial focuses on `HHTypedNeuron` and `SingleCompartment`. For neurons whose spatial structure matters, continue to the [Cell](cell.ipynb) tutorial.

## HHTypedNeuron

### Electrophysiological Properties

`HHTypedNeuron` is a fundamental abstract class in `braincell` for constructing Hodgkin–Huxley-type neuron models. In this modeling framework, the membrane potential of a neuron is determined by multiple currents:
- Active currents: Controlled by sodium and potassium channels with voltage-gated properties.
- Passive currents: Such as leak currents.
- Capacitive currents: Arising from changes in membrane potential.

This modeling approach was first proposed by Hodgkin and Huxley in 1952 to explain the mechanism of action potentials in the squid giant axon.
The HH model not only laid the foundation of modern neuronal modeling but also remains a core component of many models today.

The equivalent circuit diagram of the classical HH model is shown below:

![Equivalent circuit diagram](../_static/hh.png)

In the classical HH model, the membrane potential $V$ is governed by the following equation:

$$
C_m \frac{dV}{dt} = - (I_{\text{Na}} + I_{\text{K}} + I_L) + I_{\text{ext}}
$$

Where:
- $C_m$: Membrane capacitance
- $I_{\text{Na}}, I_{\text{K}}$: Ionic currents from sodium and potassium channels
- $I_L$: Leak current
- $I_{\text{ext}}$: External injected current (e.g., current applied through a stimulating electrode)

### Modeling Implementation

`HHTypedNeuron` inherits from:
- `Dynamics`: For defining state variables and differential equations.
- `Container`: For managing modules.
- `DiffEqModule`: A differential equation module that supports automatic modeling.

With `HHTypedNeuron`, you can flexibly combine ion channels to construct diverse dynamical configurations.
Moreover, since `HHTypedNeuron` is deeply integrated with `brainstate`, it also supports automatic differentiation and vectorized simulations, enabling batch simulations and model training.

Notably, in `braincell`, all neuron models are built upon the HH framework.
Specifically, both `SingleCompartment` and `MultiCompartment` inherit from `HHTypedNeuron`, sharing a unified interface and modeling framework. This allows us to reuse the same ion channel mechanisms and dynamical modeling strategies across neurons with different structural complexities.

## SingleCompartment

### Electrophysiological Properties

`SingleCompartment` represents a neuron model without explicit spatial structure, used for modeling conductance-based neurons with a single compartment. Its membrane potential is governed by the following differential equation:

$$
C_m \frac{dV}{dt} = \sum_j g_j(E_j - V) + I_{\text{ext}}
$$

Where:
- $C_m$: Membrane capacitance
- $V$: Membrane potential
- $g_j$: Conductance of the $j$-th ion channel
- $E_j$: Reversal potential of the $j$-th channel
- $I_{\text{ext}}$: External injected current

The conductance $g_j$ of each channel is determined by its gating variables:

$$
g_j = \bar{g}_j \cdot M^x \cdot N^y
$$

Where:
- $\bar{g}_j$: Maximum conductance
- $M$: Activation variable
- $N$: Inactivation variable
- $x, y$: Exponents depending on channel type

The gating variables follow first-order kinetics:

$$
\frac{dx}{dt} = \phi_x \cdot \frac{x_\infty(V) - x}{\tau_x(V)}
$$

Or equivalently:

$$
\frac{dx}{dt} = \phi_x \left( \alpha_x (1 - x) - \beta_x x \right)
$$

Where:
- $x \in \{M, N\}$
- $\phi_x$: Temperature factor
- $x_\infty$: Steady-state value
- $\tau_x$: Time constant
- $\alpha_x$, $\beta_x$: Rate constants

### Modeling Implementation

The implementation of `SingleCompartment` is relatively straightforward: simply inherit from `HHTypedNeuron` and define the corresponding ion channels.

Here is an example:

```python
class HH(braincell.SingleCompartment):
    def __init__(self, size, solver='rk4'):
        super().__init__(size, solver=solver)

        self.na = braincell.ion.SodiumFixed(size, E=50. * u.mV)
        self.na.add(INa=braincell.channel.Na_HH1952(size))

        self.k = braincell.ion.PotassiumFixed(size, E=-77. * u.mV)
        self.k.add(IK=braincell.channel.K_HH1952(size))

        self.IL = braincell.channel.IL(
            size,
            E=-54.387 * u.mV,
            g_max=0.03 * (u.mS / u.cm **2)
        )
```

In this example, we use `SingleCompartment` to construct a classical HH neuron with sodium (`INa`), potassium (`IK`), and leak (`IL`) currents, which together determine the electrophysiological properties of the neuron.

As this example shows, `SingleCompartment` serves as a convenient modeling interface in `braincell`, greatly simplifying the construction of neurons and improving modeling efficiency.

## Next steps

`SingleCompartment` treats the neuron as a single isopotential point. To model dendrites, axons, and the soma as a spatially connected cable system, continue to the [Cell](cell.ipynb) tutorial, which builds `MultiCompartment` models from a `Morphology`.